ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [18]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [19]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [20]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [4]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### Hour variable binary transformation
- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [24]:
from pyspark.ml.feature import SQLTransformer

#sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

In [25]:
sqlTrans = SQLTransformer(
    statement="""
    SELECT *,
           CASE
               WHEN Hour IS NULL THEN 0.0
               ELSE CAST(Hour AS DOUBLE)
           END AS Hour_double
    FROM __THIS__
    """
)

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [26]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [27]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

### One‑Hot encoding for the Month variable
Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

- We will apply `StringIndexer` before one‑hot encoding to explicitly convert a numeric column with categorical meaning into indexed categories. This prevents Spark from interpreting the values as ordinal or continuous and ensures that one‑hot encoding and downstream models treat the feature correctly as categorical.

In [28]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
#define the StringIndexer
indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_numeric",
    handleInvalid="keep"   
)

- Now we will inspect this column outside the pipeline by fitting it on the power dataset.

In [29]:
#fit only on training data
indexerTrans = indexer.fit(binary) 
#inspect results
indexerTrans.transform(binary).show(10)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+-------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|Month_numeric|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+-------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|          3.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|          3.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|          3.0|
|      6.121|    75.0|     0.083|             

In [30]:
# Month one-hot encoding
month_encoder = OneHotEncoder(
    inputCol="Month_numeric",
    outputCol="Month_ind",
    dropLast=True,
    handleInvalid="keep"
)

- We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [31]:
#inspect the one-hot encode results
#inspect results outside of pipeline
#Fit on binary data
encoder_model = month_encoder.fit(indexerTrans.transform(binary))

# Transform training data
encoded =encoder_model.transform(
    indexerTrans.transform(binary)
)

encoded.select("Month", "Month_numeric", "Month_ind").show(10)

+-----+-------------+--------------+
|Month|Month_numeric|     Month_ind|
+-----+-------------+--------------+
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
|    1|          3.0|(13,[3],[1.0])|
+-----+-------------+--------------+
only showing top 10 rows


### PAC on environmental variables 
- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [32]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures", handleInvalid="keep")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA( k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.069074321346372,0.8078678829472261] |
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.102921063880654,0.8152476222806392] |
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.1120662633791016,0.8227151924829956]|
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.1435031847422197,0.8329135817940965]|

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

### Create label column
- we will rename our response variable `Power_Zone_3` as label

In [33]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [34]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures",
    handleInvalid="keep"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("assembledFeatures", "features").show(5, truncate=False)

+------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|assembledFeatures                                                                   |features                                                                                                                                                                                                                                                                                                                                                   |
+------------------------------------------------------------------------------------+--------------------------------------------

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We are now ready to set up the pipeline, which includes transcormations and the model to be fitted. We use the `Pipeline()` function from the `pyspark.ml` module to set up a squential workflow of transformations/estimators.

In [35]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- Set up the pipeline 

In [36]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    indexer,               # Month numeric encoding
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [37]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,        #5-fold cross-validation
    parallelism=128,
    seed = 42
)
#fitting the model, and choose the best set of parameters
cv_model = cv.fit(power)

26/04/27 15:04:53 WARN Instrumentation: [2d5d0894] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:53 WARN Instrumentation: [a5d70d29] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:53 WARN Instrumentation: [9db1ddc7] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:54 WARN Instrumentation: [0769fea5] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:54 WARN Instrumentation: [b1bcad60] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:54 WARN Instrumentation: [0cf98636] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:54 WARN Instrumentation: [9fa2ee02] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:04:54 WARN Instrumentation: [da43ae82] regParam is zero, which might cause numerical instability and overf

Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [38]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [39]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.9
Best elasticNetParam: 0.5


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [40]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print(f"Best CV RMSE: {avg_rmse:.4f}")

Best CV RMSE: 2174.9888


Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [41]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)

#define evaluator
evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4505


The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [42]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "label", "prediction", "residual",).show(10, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20929.853409448464|-688.8895494484641|
|20131.08434|18698.66571942924 |1432.4186205707629|
|19668.43373|18234.524435747164|1433.9092942528368|
|18899.27711|17613.499145680427|1285.7779643195718|
|18442.40964|17015.306159163316|1427.1034808366858|
|18130.12048|16538.76719154349 |1591.3532884565102|
|17945.06024|16113.024557265051|1832.0356827349478|
|17459.27711|15740.56426361247 |1718.7128463875288|
|17025.54217|15281.325777096361|1744.2163929036396|
|16794.21687|14933.848609767041|1860.368260232959 |
+-----------+------------------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part


## Reading a stream
We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files, which created by .py file).

In [43]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

- The schema from the power DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [44]:
#set up schema
power_schema = power.schema

#set up the stream df
stream_df = spark.readStream.schema(power_schema).option("header", True).csv("csv_files")

## Transform/Aggregation Step
The streaming dataset is processed using two separate transformations derived from the same input stream, following these steps:

- The first transformation applies the fitted pipeline model to generate prediction values, residuals are computed as the difference between the observed values and the model predictions.

In [45]:
from pyspark.sql.functions import col

# Apply the fitted feature-engineering pipeline and the model to streaming data
predictions_df = (
    best_model
        .transform(stream_df)
        .withColumn("residual", col("label") - col("prediction"))     #Create a residual column
        .select("label", "prediction", "residual")
)

- The second transformation operates on the original stream and modifies the response variable `Power_Zone_3` to be named label.

In [46]:
#2nd treansformation
label_df = stream_df.withColumn("label", col("Power_Zone_3")).drop("Power_Zone_3")

- The two transformed streams are then joined on the common label column using `join()` method.

In [47]:
joined_stream = predictions_df.join(label_df, on="label")

## Write the stream

The `.writeStream` method is used to output the transformed streaming data to the console in append mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data. The five CSV files are added to the empty `csv_files` folder one at a time to be processed incrementally.

In [52]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    joined_stream
        .writeStream
        .format("console") 
        .outputMode("append")
        .start()
)

26/04/27 15:22:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fd3e00c0-7798-417d-a38f-06349569f188. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 15:22:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16196.61442|17686.098136536555|-1489.4837165365552|       25.0|   56.52|     4.902|                 0.08|        0.081| 23583.39623| 16133.47413|    8|   5|
|19299.20973|20515.125839134827|-1215.9161091348287|       19.9|    78.5|     0.069|                0.051|        0.104| 44756.58643| 25823.65145|   10|  20|
|14670.82833|15207.352650843968| -536.5243208439679|      16.95|   67.78|     0.084|                4.266|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14128.19277|16678.546326533982|-2550.3535565339826|      12.82|   69.49|     0.082|                111.9|        114.9| 30610.63291| 19524.62006|    1|  10|
|17571.49798| 18060.25616347173| -488.7581834717312|      17.94|    84.1|     0.072|                0.059|        0.126| 30115.67213| 18508.97833|    5|   0|
| 46073.9749| 36367.62634804314|  9706.348551956864|      28.63|   69.42|     4.907|                41.64|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|28547.21003|    26483.52806477|      2063.68196523|      29.88|   65.19|     4.904|                334.9|        212.5| 40281.64262| 29491.86906|    8|  16|
|9058.343337|  7445.85068383705| 1612.4926531629499|      16.54|   64.95|     0.084|                0.037|        0.185| 21669.96198| 17158.63762|   12|   5|
|15499.63636|15516.840049181852|-17.203689181851587|      14.81|    86.8|     0.073|                0.059|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16915.29648| 18092.72222282005|-1177.4257428200508|      14.44|   67.08|     0.095|                175.5|        175.2|  33724.0678|  19710.6383|    2|  15|
|16565.80645| 17472.04213831786| -906.2356883178581|      13.67|    75.4|     0.072|                104.9|         95.1| 32782.97872| 19584.14634|    3|  15|
|17571.49798| 18060.25616347173| -488.7581834717312|      17.94|    84.1|     0.072|                0.059|       

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|8350.843373|13815.053525996494|-5464.210152996495|      17.11|   69.63|     0.072|                411.8|        56.82| 31587.69231| 26743.38843|   11|  11|
|24600.50209|25821.020420273817|-1220.518330273815|      22.84|   62.29|     4.912|                0.102|        0.093|  29463.3887| 19568.35443|    7|   4|
|24845.80645|24043.730752396794| 802.0756976032062|      13.99|   64.82|     0.082|                2.075|        1.99

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|29811.15987| 27724.49923731263|  2086.66063268737|      24.87|    88.6|     4.917|                 0.08|        0.133| 37558.26859| 26291.02429|    8|   0|
|27157.66154| 27243.34661893708| -85.6850789370801|      19.22|   60.96|     0.076|                0.051|        0.085| 44910.19868| 27123.49272|    6|  21|
|32089.70711|25820.042341248154| 6269.664768751845|      31.33|   34.48|     4.907|                691.0|        34.4

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14692.34171|13342.453981971168| 1349.8877280288325|       13.8|    72.1|     0.074|                0.088|          0.1|  22557.9661| 13272.94833|    2|   2|
|23405.80645|22200.129121929014| 1205.6773280709858|      13.55|   61.68|     4.919|                0.033|        0.145| 38542.97872| 22704.87805|    3|  22|
| 16603.7247| 12799.43569022636|  3804.289009773638|      25.38|   40.97|     0.082|                429.5|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14103.87097|14133.489721547008|-29.618751547008287|       14.9|    84.2|     0.078|                0.026|        0.211| 24204.25532|  13990.2439|    3|   5|
|19618.90909|19673.639274630776|  -54.7301846307746|      18.51|   68.37|      0.07|                174.5|        168.3| 34163.18622| 19341.75153|    4|  18|
|14643.76176| 19152.69218779096|  -4508.93042779096|      24.53|    90.2|     4.918|                 0.08|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16314.21687|16589.219506167283|-275.0026361672826|      16.44|   67.16|     0.073|                385.2|        373.2| 31965.56962| 21114.89362|    1|  15|
|9185.114046| 7939.995666028693|1245.1183799713071|      11.33|   48.87|      0.08|                0.077|         0.13|  22485.1711| 18892.91194|   12|   6|
|17534.45783|15472.964920180098| 2061.492909819901|      18.55|   52.27|     0.082|                547.1|        505.

When We start the query, We read in streaming datasets using the method that created in `stream.py` file. We will submit `stream.py` in a python console.


In [51]:
# Stop the streaming query after processing all input files
query.stop()

26/04/27 15:21:34 WARN DAGScheduler: Failed to cancel job group 5464bc54-142e-4369-bae6-0138b91433bd. Cannot find active jobs for it.
26/04/27 15:21:34 WARN DAGScheduler: Failed to cancel job group 5464bc54-142e-4369-bae6-0138b91433bd. Cannot find active jobs for it.
